In [1]:
# 02_data_preparation.ipynb
import pandas as pd
import boto3
from botocore.client import Config
import io

s3 = boto3.client('s3', endpoint_url='http://minio:9000', aws_access_key_id='minioadmin', aws_secret_access_key='minioadmin123', config=Config(signature_version='s3v4'))

def read(key):
    resp = s3.get_object(Bucket='oil-data-lake', Key=key)
    return pd.read_parquet(io.BytesIO(resp['Body'].read()))

def save(df, key):
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket='oil-data-lake', Key=key, Body=buf.getvalue())

production = read('raw/production.parquet')
wells = read('raw/wells.parquet')
targets = read('raw/well_targets.parquet')
telemetry = read('raw/well_telemetry.parquet')

production['temperature'] = production['temperature'].fillna(production['temperature'].median())
production['pressure'] = production['pressure'].fillna(production['pressure'].median())

telemetry['date'] = pd.to_datetime(telemetry['timestamp']).dt.date
daily_telemetry = telemetry.groupby(['well_id', 'date']).agg({
    'pump_speed_rpm': 'mean', 'pump_current': 'mean', 'pressure_in': 'mean',
    'pressure_out': 'mean', 'temperature': 'mean', 'vibration': 'mean', 'oil_flow_rate': 'mean'
}).reset_index()
daily_telemetry.columns = ['well_id', 'date', 'avg_pump_speed_rpm', 'avg_pump_current', 'avg_pressure_in', 'avg_pressure_out', 'avg_temperature', 'avg_vibration', 'avg_oil_flow_rate']

production['downtime_ratio'] = production['downtime_hours'] / 24
production['efficiency'] = production['oil_ton'] / (production['energy_kwh'] + 1)

ml_data = production.merge(daily_telemetry, on=['well_id','date'], how='left').merge(targets, on=['well_id','date'], how='left').merge(wells[['well_id','name','status']], on='well_id', how='left')

daily_production = production.groupby('date').agg({'oil_ton':'sum', 'downtime_hours':'mean'}).reset_index()

well_kpi = production.groupby('well_id').agg({'oil_ton':['mean','sum'], 'downtime_ratio':'mean'}).round(2)
well_kpi.columns = ['avg_oil_ton', 'total_oil_ton', 'avg_downtime_ratio']
well_kpi = well_kpi.reset_index().merge(wells[['well_id','name']], on='well_id')

save(production, 'processed/production_clean.parquet')
save(ml_data, 'processed/ml_dataset.parquet')
save(daily_production, 'marts/daily_production.parquet')
save(well_kpi, 'marts/well_kpi.parquet')